# 🌾 FoodSafe AI — Biryani Rice Variety Classifier Training (Google Colab GPU)

This notebook trains the **Rice Variety Classification Deep Learning Model** for the Biryani-focused FoodSafe AI project.

### Target Classes:
1. **`basmati`**: Basmati extra-long slender rice
2. **`sona_masuri`**: Sona Masuri / HMT medium slender rice
3. **`other_rice`**: Other rice varieties (Arborio, Jasmine, Ipsala, Karacadag, Masuri, Jhili)
4. **`unknown`**: Out-of-distribution / Non-rice / Insufficient visual evidence

### Primary Dataset:
- **Source**: Mendeley Data (Prabira Sethy et al., DOI: `10.17632/c5y6gjwdzh.1`)
- **License**: CC BY 4.0
- **Content**: 8 milled rice varieties including Basmati and HMT (Sona Masuri)

### Step 1: Install Dependencies
Make sure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!pip install -q py7zr scikit-learn pillow opencv-python-headless matplotlib seaborn torchvision

### Step 2: Strict Hardware & CUDA Diagnostics
Verifies NVIDIA GPU availability before any training execution.

In [ ]:
import sys
import torch

if not torch.cuda.is_available():
    raise SystemError("CRITICAL: Compatible GPU unavailable; training would run on CPU. Please switch Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU).")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
cuda_version = torch.version.cuda
pytorch_version = torch.__version__

print(f"CUDA is available: {torch.cuda.is_available()}")
print(f"Device Name: {gpu_name}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"CUDA Version: {cuda_version}")
print(f"PyTorch Version: {pytorch_version}")

### Step 3: Dataset Acquisition, Quality Control & Leakage-Free Splitting
Downloads the verified Mendeley Data archive, filters duplicates via MD5 hashing, quarantines corrupted images, and creates stratified Train (75%), Validation (12.5%), and Test (12.5%) splits.

In [ ]:
import os
import shutil
import hashlib
import random
import urllib.request
from pathlib import Path
from PIL import Image
import py7zr

BASE_DIR = Path("./foodsafe_rice_workspace")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
EXPORT_DIR = BASE_DIR / "export"
ARCHIVE_PATH = RAW_DIR / "Milled_Rice_Dataset.7z"
EXTRACT_DIR = RAW_DIR / "extracted"

for d in [RAW_DIR, PROCESSED_DIR, EXPORT_DIR, EXTRACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOWNLOAD_URL = "https://data.mendeley.com/public-files/datasets/c5y6gjwdzh/files/86367406-086b-4e6d-a959-b6ffa63b3324/file_downloaded"

# 1. Download
if not ARCHIVE_PATH.exists() or ARCHIVE_PATH.stat().st_size < 100 * 1024 * 1024:
    print("Downloading Mendeley Data archive (152MB)...\n")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) FoodSafe-AI/1.0'}
    req = urllib.request.Request(DOWNLOAD_URL, headers=headers)
    with urllib.request.urlopen(req) as resp, open(ARCHIVE_PATH, 'wb') as out:
        total = int(resp.headers.get('Content-Length', 0))
        dl = 0
        while True:
            buf = resp.read(1024*1024)
            if not buf: break
            out.write(buf)
            dl += len(buf)
            if total: print(f"\rDownloaded: {dl/1e6:.1f}MB / {total/1e6:.1f}MB ({(dl/total)*100:.1f}%)", end="", flush=True)
    print("\nDownload complete.")
else:
    print(f"Archive already downloaded: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1e6:.1f} MB)")

# 2. Extract
if not any(EXTRACT_DIR.iterdir()):
    print("Extracting 7z archive with py7zr...")
    with py7zr.SevenZipFile(ARCHIVE_PATH, mode='r') as z:
        z.extractall(path=EXTRACT_DIR)
    print("Extraction complete.")

# 3. Quality Control & Split
TARGET_CLASSES = ["basmati", "sona_masuri", "other_rice", "unknown"]
RAW_MAP = {
    "basmati": "basmati",
    "hmt": "sona_masuri",
    "arborio": "other_rice",
    "ipsala": "other_rice",
    "jasmine": "other_rice",
    "karacadag": "other_rice",
    "masuri": "other_rice",
    "jhili": "other_rice"
}

def file_hash(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

random.seed(42)
class_images = {c: [] for c in TARGET_CLASSES}
seen_hashes = set()
corrupt_count = 0
dup_count = 0

all_files = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    all_files.extend(list(EXTRACT_DIR.rglob(ext)))

for p in all_files:
    parent_name = p.parent.name.lower()
    target = RAW_MAP.get(parent_name)
    if not target: continue
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_count += 1
        continue
    fh = file_hash(p)
    if fh in seen_hashes:
        dup_count += 1
        continue
    seen_hashes.add(fh)
    class_images[target].append(p)

# Create synthetic / non-rice images for 'unknown' class to guarantee out-of-distribution training
import numpy as np
unknown_dir = RAW_DIR / "synthetic_unknown"
unknown_dir.mkdir(exist_ok=True)
for i in range(500):
    arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    synth_p = unknown_dir / f"noise_{i}.jpg"
    Image.fromarray(arr).save(synth_p)
    class_images["unknown"].append(synth_p)

print(f"QC complete: {corrupt_count} corrupt, {dup_count} duplicates removed.")
for c, paths in class_images.items():
    print(f"  Class {c}: {len(paths)} clean unique images")

# Split into train (75%), val (12.5%), test (12.5%)
for split in ['train', 'val', 'test']:
    for c in TARGET_CLASSES:
        (PROCESSED_DIR / split / c).mkdir(parents=True, exist_ok=True)

max_per_class = 2500
for c in TARGET_CLASSES:
    paths = class_images[c]
    random.shuffle(paths)
    paths = paths[:max_per_class]
    n = len(paths)
    n_tr = int(n * 0.75)
    n_val = int(n * 0.125)
    
    for p in paths[:n_tr]: shutil.copy2(p, PROCESSED_DIR / 'train' / c / p.name)
    for p in paths[n_tr:n_tr+n_val]: shutil.copy2(p, PROCESSED_DIR / 'val' / c / p.name)
    for p in paths[n_tr+n_val:]: shutil.copy2(p, PROCESSED_DIR / 'test' / c / p.name)

print("Processed dataset structured successfully.")

### Step 4: DataLoaders & Pre-Training Print Specification
Displays the exact diagnostic information required before training initiates.

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(TARGET_CLASSES)}

train_tf = transforms.Compose([
    transforms.Resize((int(IMAGE_SIZE*1.15), int(IMAGE_SIZE*1.15))),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

class RiceDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        for c in TARGET_CLASSES:
            cdir = self.root / c
            if not cdir.exists(): continue
            idx = CLASS_TO_IDX[c]
            for p in cdir.glob("*.*"):
                self.samples.append((str(p), idx))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, lbl = self.samples[i]
        im = Image.open(p).convert("RGB")
        if self.transform: im = self.transform(im)
        return im, lbl

train_ds = RiceDataset(PROCESSED_DIR / 'train', transform=train_tf)
val_ds = RiceDataset(PROCESSED_DIR / 'val', transform=eval_tf)
test_ds = RiceDataset(PROCESSED_DIR / 'test', transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# ========================================================
# MANDATORY PRE-TRAINING SPECIFICATION DISPLAY
# ========================================================
print("--------------------------------------------------------")
print(f"GPU: {gpu_name}")
print(f"CUDA: {cuda_version}")
print(f"PyTorch: {pytorch_version}")
print("Dataset: Mendeley Data (Milled Rice Grain - Prabira Sethy DOI: 10.17632/c5y6gjwdzh.1)")
print(f"Classes: {TARGET_CLASSES}")
print(f"Train images: {len(train_ds)}")
print(f"Validation images: {len(val_ds)}")
print(f"Test images: {len(test_ds)}")
print("--------------------------------------------------------")

### Step 5: GPU Transfer Learning Training Loop
Builds EfficientNet-B0 with customized classification head, Mixed Precision (AMP), Cosine Annealing scheduler, and best-checkpoint saving.

In [ ]:
import time
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.cuda.amp import GradScaler, autocast

def build_model(num_classes=4, dropout=0.3):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_feats = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_feats, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(256, num_classes)
    )
    return m

model = build_model(num_classes=len(TARGET_CLASSES)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS = 12
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler()

best_acc = 0.0
best_state = None
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
CHECKPOINT_PATH = EXPORT_DIR / "rice_classifier.pth"

print(f"[Training] Commencing training on {gpu_name} for {EPOCHS} epochs...")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    r_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast():
            out = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        r_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total += lbls.size(0)
        correct += pred.eq(lbls).sum().item()
        
    scheduler.step()
    tr_loss = r_loss / total
    tr_acc = correct / total
    
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            with autocast():
                out = model(imgs)
                loss = criterion(out, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, pred = out.max(1)
            v_tot += lbls.size(0)
            v_corr += pred.eq(lbls).sum().item()
            
    val_loss = v_loss / v_tot
    val_acc = v_corr / v_tot
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = model.state_dict().copy()
        torch.save({
            'architecture': 'efficientnet_b0',
            'model_state_dict': best_state,
            'class_to_idx': CLASS_TO_IDX,
            'classes': TARGET_CLASSES,
            'val_acc': best_acc
        }, str(CHECKPOINT_PATH))

print(f"\n[Training] Completed in {(time.time()-t0)/60:.2f} mins. Peak Val Acc: {best_acc*100:.2f}%")

### Step 6: Test Set Evaluation, Metrics & Confusion Matrix
Evaluates on the untouched test partition.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

model.load_state_dict(best_state)
model.eval()

y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        with autocast():
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        y_true.extend(lbls.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_probs.extend(probs.cpu().numpy().tolist())

y_true, y_pred = np.array(y_true), np.array(y_pred)
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
p_cls, r_cls, f_cls, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=list(range(len(TARGET_CLASSES))), zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(TARGET_CLASSES))))

print("========================================================")
print(f"TEST ACCURACY:      {acc*100:.2f}%")
print(f"WEIGHTED PRECISION: {prec:.4f}")
print(f"WEIGHTED RECALL:    {rec:.4f}")
print(f"WEIGHTED F1-SCORE:  {f1:.4f}")
print("--------------------------------------------------------")
for i, c in enumerate(TARGET_CLASSES):
    print(f"  {c.upper():<12} -> Precision: {p_cls[i]:.4f} | Recall: {r_cls[i]:.4f} | F1: {f_cls[i]:.4f}")
print("========================================================")

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
plt.title('Rice Variety Classification - Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('Ground Truth')
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'confusion_matrix.png')
plt.show()

### Step 7: Export Model Artifacts & Download Zip for FoodSafe AI
Saves all artifacts strictly conforming to the FoodSafe AI inference service specification and automatically packages them for 1-click download.

In [ ]:
import zipfile
from google.colab import files

# 1. Save Best Checkpoint
torch.save({
    'architecture': 'efficientnet_b0',
    'model_state_dict': best_state,
    'class_to_idx': CLASS_TO_IDX,
    'classes': TARGET_CLASSES,
    'test_accuracy': float(acc),
    'test_f1': float(f1),
    'image_size': IMAGE_SIZE,
    'norm_mean': NORM_MEAN,
    'norm_std': NORM_STD,
    'confidence_threshold': 0.65
}, str(EXPORT_DIR / 'rice_classifier.pth'))

# 2. Save class_names.json
with open(EXPORT_DIR / 'class_names.json', 'w') as f:
    json.dump({str(i): c for i, c in enumerate(TARGET_CLASSES)}, f, indent=2)

# 3. Save preprocessing_config.json
with open(EXPORT_DIR / 'preprocessing_config.json', 'w') as f:
    json.dump({
        'architecture': 'efficientnet_b0',
        'image_size': IMAGE_SIZE,
        'normalization': {'mean': NORM_MEAN, 'std': NORM_STD},
        'classes': TARGET_CLASSES,
        'confidence_threshold': 0.65,
        'hardware': {'gpu': gpu_name, 'cuda': cuda_version, 'pytorch': pytorch_version}
    }, f, indent=2)

# 4. Save metrics.json
with open(EXPORT_DIR / 'metrics.json', 'w') as f:
    json.dump({
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
        'confusion_matrix': cm.tolist(),
        'history': history
    }, f, indent=2)

# 5. Zip Artifacts
zip_filename = "foodsafe_rice_model.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file_name in ['rice_classifier.pth', 'class_names.json', 'preprocessing_config.json', 'metrics.json', 'confusion_matrix.png']:
        file_path = EXPORT_DIR / file_name
        if file_path.exists():
            zipf.write(file_path, arcname=file_name)

print(f"\nAll artifacts successfully packaged into {zip_filename}!")
print("Downloading model bundle to your local computer...")
files.download(zip_filename)